# 9주차 실습. 작은 확산 모형을 처음부터 끝까지
*Week 9 lab. A small diffusion model, from noise to data and back*

**제출하지 않습니다. 채점하지 않습니다.** 과제 3 의 출발점입니다(과제 3 은 11/11 에 따로 나옵니다. 손대지 않은 이 노트북은 과제 3 제출로 치지 않습니다).
*Nothing to submit, nothing graded. This is the starting point for Assignment 3 (released separately on 11/11; an unchanged lab notebook does not count as a submission).*

**이 실습의 목표 한 문장.** 네 군집의 2차원 점들을 정해진 요리법대로 잡음으로 녹인 뒤, 그 한 줄 공식(어느 걸음이든 바로 건너뛰기)과 역방향 한 걸음 공식을 직접 쓰고, 그 두 줄로 작은 DDPM 을 학습 · 표집하여 손실이 내려갈 때 표본이 어떻게 달라지는지, 걸음을 줄이면 무엇을 잃는지 봅니다.
*Goal in one sentence. Melt 2-D points from four clusters into noise by a fixed recipe, write the one-line jump to any step and the one-line reverse step yourself, then train and sample a tiny DDPM with those two lines to see how the samples change as the loss falls and what is lost when you take fewer steps.*

| 절 / Section | 하는 일 / What you do | 시간 / Time |
|---|---|---|
| §0 | 준비. 쉬는 시간에 실행하세요(내려받을 것 없음)<br>Setup. Run it during the break (nothing to download) | 쉬는 시간 / during the break |
| 몸풀기 / Warm-up | 슬라이드의 한 픽셀 숫자를 코드로 확인. 빈칸 없음<br>Check the one-pixel numbers from the slides in code. No blank | 2분 / 2 min |
| §1 | 네 군집을 한 걸음씩 잡음으로 녹이기. 빈칸 없음<br>Melt the four clusters into noise one step at a time. No blank | 3분 / 3 min |
| §2 | 빈칸 1. 어느 걸음이든 한 줄로 건너뛰기<br>Blank 1. Jump to any step in one line | 6분 / 6 min |
| §3 | 작은 DDPM 학습(제공). 손실이 어디까지 내려가는지 예측<br>Train a tiny DDPM (provided). Predict how far the loss falls | 4분 / 4 min |
| §4 | 빈칸 2. 역방향 한 걸음, 그리고 표집<br>Blank 2. One reverse step, then sampling | 8분 / 8 min |
| §5 | DDIM 으로 걸음 줄이기. 어디서 꺾이는지 찾기<br>DDIM with fewer steps. Find where quality bends | 5분 / 5 min |
| 선택 / Optional | 흐름 매칭 열 줄<br>The flow-matching recipe in ten lines | 집에서 / at home |
| 정리 / Wrap-up | 예측 다시 읽기<br>Re-read your predictions | 2분 / 2 min |

수업 중 실습은 30분입니다. 선택 절은 시간이 남을 때나 집에서 하세요.
*The lab gets 30 minutes in class. Do the optional section if time is left, or at home.*

슬라이드와 짝(9주차 본 덱): 몸풀기 「The Jump, on One Pixel」 · 「One Training Step by Hand」 · 「The Network Learns to Predict the Noise」, §1 「The Forward Process Adds Noise by a Fixed Recipe」, §2 「Jump to Any Step in One Line」, §3 「The Loss, Words First」 · 「Why Is DDPM Often Easier to Train Than a GAN?」 · 「As the Loss Falls, Do the Samples Match the Data?」, §4 「One DDPM Reverse Step」 · 「The Same Step, in Numbers」, §5 「DDIM: Can We Generate With Fewer Steps?」 · 「What Happens When We Skip Steps?」, 선택 「Optional D · Flow Matching」, 함정 「Four Ways to Get This Wrong」.
*Matching slides in the Week 9 main deck, section by section, with the same titles.*

**GPU 가 없어도 됩니다.** 2코어 CPU 에서 §3 학습이 10초 안팎, 나머지 셀은 몇 초 안쪽입니다. 슬라이드의 그림 세 장(학습 곡선과 표본, DDIM 걸음 수, 베끼기)은 이 노트북과 같은 설정 · 같은 시드로 만든 것이라, 같은 컴퓨터에서는 같은 숫자가 나옵니다.
*No GPU needed. On a 2-core CPU the §3 training takes about 10 seconds; every other cell takes a few seconds at most. Three figures in the deck (loss curve and samples, DDIM step counts, copying) were made with this notebook's setup and seeds, so the same computer gives the same numbers.*

## §0 준비 / Setup
아래 셀을 실행하면 라이브러리를 불러오고, 오늘 내내 쓸 **잡음 요리법**(걸음마다 더할 잡음의 크기 $\beta_t$ 와 그 누적 $\bar\alpha_t$)을 만듭니다. 내려받을 것은 없습니다. **쉬는 시간에 실행해 두세요.**
*Run the cell below. It imports the libraries and builds the **noise recipe** we use all lab long: the per-step noise size $\beta_t$ and its running product $\bar\alpha_t$. Nothing to download. **Run it during the break.***

먼저 **파일 → Drive에 사본 저장**을 누르세요. 그러지 않으면 작업 내용이 사라집니다.
*First, File → Save a copy in Drive. Otherwise your work is lost.*

슬라이드의 DDPM 은 T = 1000 걸음에 $\beta_t$ 를 0.0001 에서 0.02 까지 키웁니다. 여기서는 걸음을 100 으로 줄이는 대신 $\beta_t$ 를 0.0008 에서 0.08 까지 키워, 끝에서 남는 신호 $\sqrt{\bar\alpha_T}$ 가 슬라이드처럼 거의 0 이 되게 합니다. 걸음이 적어야 표집이 몇 초에 끝납니다.
*The DDPM on the slides takes T = 1000 steps with $\beta_t$ rising from 0.0001 to 0.02. Here we take 100 steps and raise $\beta_t$ from 0.0008 to 0.08 instead, so that the surviving signal $\sqrt{\bar\alpha_T}$ at the end is almost 0, as on the slides. Fewer steps keep sampling to a few seconds.*

In [ ]:
import math, time, copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_num_threads(2)   # 스레드 수 고정. 같은 컴퓨터에서 다시 돌리면 같은 숫자가 나온다 / fix the thread count so a re-run gives the same numbers

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

GREEN, GOLD, RED, GRAY = "#406551", "#C9971C", "#B04A3A", "#9AA5A0"   # 그림 색 / figure colors

# 잡음 요리법. t = 1 … T 가 슬라이드의 걸음 번호이고, 0 번 칸은 「잡음을 더하기 전」(x_0 = 데이터) 을 뜻하는 자리입니다.
# The noise recipe. t = 1 … T are the step numbers on the slides; slot 0 stands for "before any noise" (x_0 = the data).
T = 100
betas = torch.cat([torch.zeros(1), torch.linspace(0.0008, 0.08, T)])   # β_0 = 0, β_1 … β_T 가 직선으로 커진다 / β_0 = 0, β_1 … β_T rise linearly
alphas = 1 - betas                                                      # α_t = 1 − β_t
abar = torch.cumprod(alphas, 0)                                         # ᾱ_t = α_1 × … × α_t. ᾱ_0 = 1 / the running product; ᾱ_0 = 1

print("t   β_t      ᾱ_t      남는 신호 √ᾱ_t / surviving signal")
for t in (1, 10, 25, 50, 75, 100):
    print(f"{t:3d} {betas[t]:.4f}  {abar[t]:.4f}  {abar[t].sqrt():.3f}")
# 기대값 / Expected
#   t   β_t      ᾱ_t      남는 신호 √ᾱ_t / surviving signal
#     1 0.0008  0.9992  1.000
#    10 0.0080  0.9568  0.978
#    25 0.0200  0.7697  0.877
#    50 0.0400  0.3556  0.596
#    75 0.0600  0.0976  0.312
#   100 0.0800  0.0157  0.125
#   ᾱ_t 가 1 에서 0 가까이로 내려갑니다. 슬라이드의 표(T = 1000)와 같은 모양이고, 걸음 수만 다릅니다.
#   ᾱ_t runs from 1 down toward 0: the same shape as the table on the slide (T = 1000), only with fewer steps.

## 몸풀기. 슬라이드의 한 픽셀 / Warm-up. The one pixel from the slides
「The Jump, on One Pixel」 과 「One Training Step by Hand」 의 숫자를 그대로 씁니다. 픽셀 하나 $x_0 = 2$, 그 걸음의 $\bar\alpha_t = 0.36$, 뽑은 잡음 $\varepsilon = -1.5$. 빈칸은 없습니다. 숫자가 슬라이드와 같은지만 보세요.
*The numbers from those two slides, as they are: one pixel $x_0 = 2$, that step's $\bar\alpha_t = 0.36$, a drawn noise $\varepsilon = -1.5$. No blank. Just check that the numbers match the slides.*

**코드를 돌리기 전에** 슬라이드에서 두 답을 찾아 말해 보세요.
*Before running, find two answers on the slides and say them.*

1. $x_t$ 는 얼마인가요? 신호에 붙는 계수와 잡음에 붙는 계수는? / What is $x_t$, and what are the two coefficients, on the signal and on the noise?
2. 신경망이 $\hat\varepsilon = -1.2$ 라고 답하면 손실은? 그 $\hat\varepsilon$ 로 되돌린 $\hat x_0$ 는 2 에서 얼마나 벗어나나요? (「The Network Learns to Predict the Noise」)
   *If the network answers $\hat\varepsilon = -1.2$, what is the loss, and how far from 2 is the $\hat x_0$ recovered from that $\hat\varepsilon$?*

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   x0      : 깨끗한 픽셀 값 / the clean pixel value
#   abar_t  : 이 걸음에서 남은 신호의 비율 ᾱ_t / the surviving share of signal at this step, ᾱ_t
#   eps     : 섞은 잡음 ε / the noise ε that was mixed in
#   x_t     : 잡음이 섞인 값 / the noisy value
#   eps_hat : 신경망의 답 ε̂ / the network's answer ε̂
#   x0_hat  : ε̂ 로 되돌린 x̂0 / x̂0 recovered from ε̂

x0, abar_t, eps = 2.0, 0.36, -1.5
x_t = math.sqrt(abar_t) * x0 + math.sqrt(1 - abar_t) * eps          # 「The Jump, on One Pixel」 의 한 줄 / the one line on that slide
x_t = round(x_t, 6) or 0.0                                            # 컴퓨터의 −0.0000000000000002 를 0.0 으로 / turn the computer's −0.0000000000000002 into 0.0
print(f"계수 / coefficients: 신호 {math.sqrt(abar_t):.1f}, 잡음 {math.sqrt(1 - abar_t):.1f} / signal {math.sqrt(abar_t):.1f}, noise {math.sqrt(1 - abar_t):.1f}")
print(f"x_t = {x_t:.1f}")

eps_hat = -1.2
loss = (eps - eps_hat) ** 2                                           # 「One Training Step by Hand」 5번 줄 / line 5 of that slide
x0_hat = (x_t - math.sqrt(1 - abar_t) * eps_hat) / math.sqrt(abar_t)  # 「The Network Learns to Predict the Noise」 의 식 / the formula on that slide
print(f"loss = {loss:.2f}")
print(f"x̂0 from ε̂ = -1.2 : {x0_hat:.1f}    (참값 2 와의 차이 / off from the true 2 by {abs(x0_hat - 2):.1f})")
# 기대값 / Expected
#   계수: 신호 0.6, 잡음 0.8        잡음 쪽이 이미 더 무겁습니다 / the noise already carries more weight
#   x_t = 0.0                      슬라이드와 같음 / as on the slide
#   loss = 0.09                    (−1.5 − (−1.2))² / squared error
#   x̂0 from ε̂ = -1.2 : 1.6         잡음 오차 0.3 이 0.8/0.6 배인 0.4 로 커졌습니다. 이 걸음에서는 잡음의 작은 실수가 픽셀의 더 큰 실수가 됩니다.
#                                  the noise error 0.3 grew by 0.8/0.6 to 0.4: at this step a small slip in the noise is a bigger slip in the pixel

## §1 네 군집을 잡음으로 녹이기 / Melt the four clusters into noise
오늘의 데이터는 2차원 점 2,000개입니다. 네 군집의 중심은 $(\pm 2, \pm 2)$ 이고, 각 점은 중심 둘레에 조금 흩어져 있습니다. 이미지 대신 2차원 점을 쓰는 이유는 하나입니다. **모든 것을 그림으로 볼 수 있습니다.**
*Today's data are 2,000 points in 2-D. The four cluster centers are $(\pm 2, \pm 2)$ and each point scatters a little around its center. We use 2-D points instead of images for one reason: **everything can be seen in a picture.***

「The Forward Process Adds Noise by a Fixed Recipe」 의 한 걸음은 다음과 같습니다. 신호를 조금 줄이고 작은 가우스 잡음을 더합니다. 배우는 것은 없습니다.
*One step of that slide: shrink the signal slightly and add a little Gaussian noise. Nothing is learned.*

$$x_t = \sqrt{1 - \beta_t}\, x_{t-1} + \sqrt{\beta_t}\, \varepsilon_t, \qquad \varepsilon_t \sim N(0, I)$$

**코드를 돌리기 전에** 예측을 적으세요. 100 걸음 중 몇 걸음쯤이면 네 군집이 더 이상 보이지 않을까요?
*Before running, write a prediction: after about how many of the 100 steps will the four clusters no longer be visible?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_1 = ""

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   CENTERS : 네 군집의 중심 (±2, ±2), 크기 (4, 2) / the four cluster centers, shape (4, 2)
#   SPREAD  : 중심 둘레의 흩어짐(표준편차) / the scatter around a center (standard deviation)
#   data    : 학습 점 2,000개, 크기 (2000, 2) / the 2,000 training points, shape (2000, 2)
#   x       : 지금까지 잡음을 더한 점들. 처음엔 data 와 같다 / the points after the steps so far; starts equal to data
#   snaps   : 몇 걸음에서의 x 를 그림용으로 담아 둔 사전 / x at a few steps, kept for the figure

CENTERS = torch.tensor([[-2., -2.], [-2., 2.], [2., -2.], [2., 2.]])
SPREAD = 0.24

def make_data(n, gen):
    """중심 넷 중 하나를 고르고 그 둘레에 흩뿌린 점 n 개 / n points: pick one of the four centers, scatter around it."""
    ids = torch.randint(0, 4, (n,), generator=gen)                    # 점마다 군집 번호 / a cluster index per point
    return CENTERS[ids] + torch.randn(n, 2, generator=gen) * SPREAD    # 중심 + 작은 흩어짐 / center plus a small scatter

data = make_data(2000, torch.Generator().manual_seed(29))             # 데이터 난수 고정 / fix the data's randomness

gen_fwd = torch.Generator().manual_seed(1)                            # 걸음마다 더하는 잡음의 난수 고정 / fix the per-step noise
x = data.clone()                                                      # t = 0: 데이터 그대로 / t = 0: the data itself
snaps = {0: x.clone()}
for t in range(1, T + 1):                                             # t = 1 … T, 한 걸음씩 / one step at a time
    eps_t = torch.randn(x.shape, generator=gen_fwd)                   # 이 걸음의 새 잡음 / new noise for this step
    x = torch.sqrt(1 - betas[t]) * x + torch.sqrt(betas[t]) * eps_t   # 슬라이드의 한 걸음 / the one step on the slide
    if t in (10, 25, 50, 100):
        snaps[t] = x.clone()

fig, axs = plt.subplots(1, 5, figsize=(11, 2.5))
for ax, t in zip(axs, (0, 10, 25, 50, 100)):
    ax.scatter(snaps[t][:, 0], snaps[t][:, 1], s=2, color=GREEN, alpha=0.5)
    ax.set(xlim=(-5, 5), ylim=(-5, 5), xticks=[-4, 0, 4], yticks=[-4, 0, 4], title=f"t = {t}, √ᾱ = {abar[t].sqrt():.2f}")
    ax.set_aspect("equal")
plt.suptitle("Four clusters melting into noise, one step at a time", fontsize=10); plt.tight_layout(); plt.show()

print("t    점들의 퍼짐(표준편차) / spread of the points (standard deviation)   요리법이 말하는 값 / what the recipe says")
for t in (0, 10, 25, 50, 100):
    said = math.sqrt(abar[t] * data.var(0).mean().item() + (1 - abar[t]))   # √(ᾱ_t · 데이터 분산 + (1 − ᾱ_t)) / √(ᾱ_t · data variance + (1 − ᾱ_t))
    print(f"{t:3d}  {snaps[t].std(0).mean():.3f}                                                 {said:.3f}")
# 기대값 / Expected   (왼쪽은 실제 점들, 오른쪽은 식으로 구한 값. 둘이 거의 같아야 합니다 / left: the actual points, right: from the formula. They should nearly agree)
#     0  2.013   2.013       데이터 자체. 중심이 ±2 라 퍼짐이 약 2 / the data itself; centers at ±2 give a spread of about 2
#    10  1.979   1.980
#    25  1.836   1.830
#    50  1.448   1.444
#   100  1.032   1.024       거의 N(0, 1). 군집은 사라졌습니다 / nearly N(0, 1); the clusters are gone

**무엇을 보았나.** 걸음마다 신호는 $\sqrt{1-\beta_t}$ 배로 조금 줄고 잡음이 조금 더해집니다. 50 걸음쯤이면 네 군집이 흐려지고 100 걸음이면 그냥 둥근 잡음입니다. 이 과정에 배우는 것은 없습니다. 함정 1(「Four Ways to Get This Wrong」): 모형은 잡음을 더하는 법을 배우지 않습니다. 더하는 법은 우리가 정했고, 배우는 것은 지우는 쪽뿐입니다.
*What did we see. Each step shrinks the signal by $\sqrt{1-\beta_t}$ and adds a little noise. By about step 50 the four clusters blur, and at step 100 it is just round noise. Nothing here is learned. Pitfall 1 on that slide: the model does not learn to add noise; we chose how to add it, and only the removing is learned.*

## §2 빈칸 1. 어느 걸음이든 한 줄로 건너뛰기 / Blank 1. Jump to any step in one line
§1 처럼 걸음을 하나씩 밟으면 $x_{50}$ 을 얻는 데 50번 잡음을 뽑아야 합니다. 학습에서는 걸음 t 를 무작위로 고르고 그 $x_t$ 가 바로 필요하므로, 이래서는 느립니다.
*Stepping one at a time as in §1, getting $x_{50}$ takes 50 noise draws. Training picks a random step t and needs that $x_t$ right away, so this is too slow.*

「Jump to Any Step in One Line」 이 답입니다. 가우스 잡음을 여러 번 더한 것은 가우스 잡음 한 번과 같으므로, $x_0$ 에서 $x_t$ 로 한 줄에 갑니다. 신호에는 $\sqrt{\bar\alpha_t}$, 잡음에는 $\sqrt{1 - \bar\alpha_t}$ 가 붙습니다. 몸풀기에서 숫자로 본 바로 그 줄입니다.
*That slide is the answer. Several Gaussian noises added up are one Gaussian noise, so we go from $x_0$ to $x_t$ in one line: $\sqrt{\bar\alpha_t}$ on the signal, $\sqrt{1 - \bar\alpha_t}$ on the noise. The very line you saw in numbers in the warm-up.*

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \varepsilon, \qquad \varepsilon \sim N(0, I)$$

함정 2 를 조심하세요. 한 줄 공식에 들어가는 것은 걸음 하나의 $\beta_t$ 가 아니라 누적 곱 $\bar\alpha_t$ 입니다.
*Mind pitfall 2: the one-line formula takes the running product $\bar\alpha_t$, not the single-step $\beta_t$.*

**빈칸 하나.** $x_0$, $\varepsilon$, $\bar\alpha_t$ 를 받아 $x_t$ 를 돌려주는 함수입니다. §3 의 학습이 매 걸음 이 함수를 그대로 씁니다.
*The blank. A function that takes $x_0$, $\varepsilon$, $\bar\alpha_t$ and returns $x_t$. The training in §3 calls this very function at every step.*

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   x0      : 깨끗한 점들, 크기 (점 수, 2) / the clean points, shape (number of points, 2)
#   eps     : 점마다 뽑은 잡음 ε, x0 와 같은 크기 / the noise ε drawn for each point, same shape as x0
#   abar_t  : 점마다의 ᾱ_t, 크기 (점 수, 1). 행마다 곱해진다 / each point's ᾱ_t, shape (number of points, 1); multiplies row by row
#   x_t     : 잡음이 섞인 점들   <- 여기가 빈칸
#             the noisy points   <- this is the blank
#
# 참고 / Reference   torch.sqrt(a)
#   a 의 칸마다 제곱근을 돌려준다 / the square root of every entry of a
#   예 / e.g.   torch.sqrt(torch.tensor([[0.25], [1.0]]))  ->  tensor([[0.5000], [1.0000]])
#   (점 수, 1) 크기의 텐서에 (점 수, 2) 크기를 곱하면 행마다 같은 수가 두 칸에 곱해진다.
#   Multiplying a (points, 1) tensor by a (points, 2) tensor multiplies both entries of each row by that row's number.

def jump(x0, eps, abar_t):
    """x_0 에서 걸음 t 의 x_t 로 한 줄에 간다 / Go from x_0 to the x_t of step t in one line."""
    # TODO. 슬라이드 「Jump to Any Step in One Line」 의 식을 그대로 쓰세요.
    #       신호에 붙는 계수와 잡음에 붙는 계수, 둘 다 ᾱ_t 에서 나옵니다. 한 줄입니다.
    # TODO. Write the formula from the slide "Jump to Any Step in One Line" as it is.
    #       Both coefficients, on the signal and on the noise, come from ᾱ_t. One line.
    x_t = None

    if x_t is None:
        raise NotImplementedError(BLANK_MSG)

    return x_t

# 자가검증 / Check your work
# 1. 몸풀기의 픽셀: x0 = 2, ᾱ = 0.36, ε = −1.5 / the warm-up pixel
print(jump(torch.tensor([[2.0]]), torch.tensor([[-1.5]]), torch.tensor([[0.36]])))
# 2. 점 하나, 두 좌표: x0 = (−2, 2), ε = (0.5, −0.5), ᾱ = 0.25 / one point, two coordinates
print(jump(torch.tensor([[-2.0, 2.0]]), torch.tensor([[0.5, -0.5]]), torch.tensor([[0.25]])))
# 기대값 / Expected
#   tensor([[0.]])                      몸풀기와 같음 / as in the warm-up
#   tensor([[-0.5670,  0.5670]])        0.5 × (−2) + 0.866 × 0.5 = −0.567
#   tensor([[-0.2400]]) 이면 ᾱ_t 에 제곱근을 씌우지 않았습니다. 계수는 √ᾱ_t 와 √(1 − ᾱ_t) 입니다.
#   tensor([[-0.2400]]) means the square roots were left off ᾱ_t. The coefficients are √ᾱ_t and √(1 − ᾱ_t).
#   tensor([[0.2400]]) 이면 잡음 쪽 계수에만 제곱근이 빠졌습니다.
#   tensor([[0.2400]]) means the square root was left off the noise coefficient only.
#   tensor([[0.7000]]) 이면 두 계수가 서로 바뀌었습니다. 신호에 √ᾱ_t, 잡음에 √(1 − ᾱ_t) 입니다.
#   tensor([[0.7000]]) means the two coefficients were swapped: √ᾱ_t goes on the signal, √(1 − ᾱ_t) on the noise.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요. 이미 채웠다면 실행하지 마세요(내 함수를 덮어씁니다) / Only if stuck. If you already filled in the blank, skip it: it overwrites your function { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

# 다음 이름을 씁니다 / Names used below
#   x0      : 깨끗한 점들, 크기 (점 수, 2) / the clean points, shape (number of points, 2)
#   eps     : 점마다 뽑은 잡음 ε, x0 와 같은 크기 / the noise ε drawn for each point, same shape as x0
#   abar_t  : 점마다의 ᾱ_t, 크기 (점 수, 1). 행마다 곱해진다 / each point's ᾱ_t, shape (number of points, 1); multiplies row by row
#   x_t     : 잡음이 섞인 점들   <- 여기가 빈칸
#             the noisy points   <- this is the blank
#
# 참고 / Reference   torch.sqrt(a)
#   a 의 칸마다 제곱근을 돌려준다 / the square root of every entry of a
#   예 / e.g.   torch.sqrt(torch.tensor([[0.25], [1.0]]))  ->  tensor([[0.5000], [1.0000]])
#   (점 수, 1) 크기의 텐서에 (점 수, 2) 크기를 곱하면 행마다 같은 수가 두 칸에 곱해진다.
#   Multiplying a (points, 1) tensor by a (points, 2) tensor multiplies both entries of each row by that row's number.

def jump(x0, eps, abar_t):
    """x_0 에서 걸음 t 의 x_t 로 한 줄에 간다 / Go from x_0 to the x_t of step t in one line."""
    x_t = torch.sqrt(abar_t) * x0 + torch.sqrt(1 - abar_t) * eps   # 빈칸이었던 줄. 신호에 √ᾱ_t, 잡음에 √(1 − ᾱ_t)
                                                                    # the blank: √ᾱ_t on the signal, √(1 − ᾱ_t) on the noise

    if x_t is None:
        raise NotImplementedError(BLANK_MSG)

    return x_t

# 자가검증 / Check your work
# 1. 몸풀기의 픽셀: x0 = 2, ᾱ = 0.36, ε = −1.5 / the warm-up pixel
print(jump(torch.tensor([[2.0]]), torch.tensor([[-1.5]]), torch.tensor([[0.36]])))
# 2. 점 하나, 두 좌표: x0 = (−2, 2), ε = (0.5, −0.5), ᾱ = 0.25 / one point, two coordinates
print(jump(torch.tensor([[-2.0, 2.0]]), torch.tensor([[0.5, -0.5]]), torch.tensor([[0.25]])))
# 기대값 / Expected
#   tensor([[0.]])                      몸풀기와 같음 / as in the warm-up
#   tensor([[-0.5670,  0.5670]])        0.5 × (−2) + 0.866 × 0.5 = −0.567
#   tensor([[-0.2400]]) 이면 ᾱ_t 에 제곱근을 씌우지 않았습니다. 계수는 √ᾱ_t 와 √(1 − ᾱ_t) 입니다.
#   tensor([[-0.2400]]) means the square roots were left off ᾱ_t. The coefficients are √ᾱ_t and √(1 − ᾱ_t).
#   tensor([[0.2400]]) 이면 잡음 쪽 계수에만 제곱근이 빠졌습니다.
#   tensor([[0.2400]]) means the square root was left off the noise coefficient only.
#   tensor([[0.7000]]) 이면 두 계수가 서로 바뀌었습니다. 신호에 √ᾱ_t, 잡음에 √(1 − ᾱ_t) 입니다.
#   tensor([[0.7000]]) means the two coefficients were swapped: √ᾱ_t goes on the signal, √(1 − ᾱ_t) on the noise.

In [ ]:
# 한 줄 건너뛰기와 §1 의 한 걸음씩이 같은 분포를 주는지 봅니다. 점마다의 값은 잡음을 다르게 뽑아 다르지만, 퍼짐은 같아야 합니다.
# Does the one-line jump give the same distribution as the step-by-step walk of §1? Individual points differ (different noise draws), but the spread must agree.
gen_jump = torch.Generator().manual_seed(2)
print("t    한 걸음씩 / step by step    한 줄로 / one line")
for t in (10, 25, 50, 100):
    eps = torch.randn(data.shape, generator=gen_jump)                       # 점마다 잡음 한 번 / one noise draw per point
    x_jump = jump(data, eps, abar[t].expand(len(data), 1))                   # 모든 점에 같은 ᾱ_t / the same ᾱ_t for every point
    print(f"{t:3d}  {snaps[t].std(0).mean():.3f}                      {x_jump.std(0).mean():.3f}")
# 기대값 / Expected   두 열이 소수 둘째 자리까지 거의 같습니다 / the two columns agree to about two decimals
#    10  1.979   1.979
#    25  1.836   1.834
#    50  1.448   1.426
#   100  1.032   1.035

fig, axs = plt.subplots(1, 2, figsize=(5.2, 2.6))
for ax, pts, title in zip(axs, (snaps[50], jump(data, torch.randn(data.shape, generator=gen_jump), abar[50].expand(len(data), 1))), ("t = 50, step by step", "t = 50, one line")):
    ax.scatter(pts[:, 0], pts[:, 1], s=2, color=GREEN, alpha=0.5)
    ax.set(xlim=(-5, 5), ylim=(-5, 5), xticks=[-4, 0, 4], yticks=[-4, 0, 4], title=title); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

## §3 작은 DDPM 학습 (제공) / Train a tiny DDPM (provided)
배우는 것은 하나뿐입니다. $x_t$ 와 t 를 받아 그 안에 섞인 잡음 $\varepsilon$ 을 맞히는 신경망 $\varepsilon_\theta(x_t, t)$. 손실은 「The Loss, Words First」 그대로, 아는 잡음을 섞고 그 잡음을 물어 제곱 오차를 재는 것입니다. 아래 학습 셀의 한 걸음은 몸풀기의 숫자 계산을 점 512개에 한꺼번에 한 것이고, $x_t$ 는 방금 쓴 `jump` 가 만듭니다.
*Only one thing is learned: a network $\varepsilon_\theta(x_t, t)$ that takes $x_t$ and t and guesses the noise $\varepsilon$ mixed into it. The loss is that slide as it is: mix in known noise, ask for it back, square the error. One step of the training cell below is the warm-up calculation done on 512 points at once, and the $x_t$ comes from your `jump`.*

신경망은 작은 MLP 입니다. 입력은 점의 두 좌표와 걸음 t 를 사인 · 코사인으로 편 16개 숫자(t 가 100 이라는 큰 수 하나로 들어가면 학습이 어렵습니다), 출력은 잡음의 두 좌표입니다.
*The network is a small MLP. Its input is the point's two coordinates plus the step t spread into 16 sine and cosine numbers (one big number like 100 is hard to learn from); its output is the two coordinates of the noise.*

**코드를 돌리기 전에** 예측을 적으세요. 손실(잡음의 제곱 오차)은 0 까지 내려갈까요? 내려가지 않는다면 왜일까요?
*Before running, write a prediction: will the loss (the squared error on the noise) fall all the way to 0? If not, why not?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_2 = ""

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   Net        : 잡음 예측기 ε_θ(x_t, t). 입력 2 + 16, 은닉 64 × 2, 출력 2 / the noise predictor; input 2 + 16, hidden 64 × 2, output 2
#   t_features : 걸음 t 를 사인 · 코사인 16개로 펴는 함수 / spreads the step t into 16 sines and cosines
#   model      : 학습할 신경망 / the network we train
#   opt        : 가중치를 고치는 규칙(Adam, 경사하강법의 한 종류) / the update rule (Adam, a kind of gradient descent)
#   losses     : 걸음마다의 손실 / the loss at every update
#   ckpt       : 150 · 600 · 2,000 걸음에서의 가중치 사본. §4 에서 표집에 쓴다 / copies of the weights at updates 150, 600, 2,000, used for sampling in §4

def t_features(t):
    """t = 1 … T 를 [0, 1) 로 줄인 뒤 주파수 8개의 사인 · 코사인으로 편다 / scale t = 1 … T into [0, 1), then spread it over 8 frequencies of sine and cosine."""
    tt = (t.float()[:, None] - 1) / T
    freqs = (2 ** torch.arange(8).float())[None, :]
    return torch.cat([torch.sin(tt * freqs * math.pi), torch.cos(tt * freqs * math.pi)], 1)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(18, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 2))

    def forward(self, x, t):
        return self.net(torch.cat([x, t_features(t)], 1))              # 좌표 2개 + t 의 16개 숫자 → 잡음 2개 / 2 coordinates + 16 numbers for t → 2 noise entries

gen = torch.Generator().manual_seed(29)                                  # 미니배치 · t · ε 의 난수 고정 / fix the draws of minibatch, t and ε
torch.manual_seed(29)                                                    # 신경망 초기값 고정 / fix the initial weights
model = Net()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses, ckpt = [], {}
t0 = time.time()
for step in range(1, 2001):
    idx = torch.randint(0, len(data), (512,), generator=gen)              # 학습 점 512개를 무작위로 / 512 random training points
    x0 = data[idx]
    t = torch.randint(1, T + 1, (len(x0),), generator=gen)              # 점마다 걸음 t 를 무작위로 / a random step t per point
    eps = torch.randn(x0.shape, generator=gen)                           # 점마다 새 잡음 / new noise per point
    x_t = jump(x0, eps, abar[t][:, None])                                # §2 의 한 줄 / the one line from §2
    loss = (model(x_t, t) - eps).square().mean()                         # 「The Loss, Words First」: 잡음 맞히기의 제곱 오차 / squared error on the noise
    opt.zero_grad()                                                      # 지난 걸음의 기울기를 지운다 / clear last step's gradients
    loss.backward()                                                      # 기울기 계산 / compute the gradients
    opt.step()                                                           # 가중치 한 걸음 / one update of the weights
    losses.append(loss.item())
    if step in (150, 600, 2000):
        ckpt[step] = copy.deepcopy(model.state_dict())                   # 이 시점의 가중치를 담아 둔다 / keep a copy of the weights now
print(f"처음 100걸음 평균 손실 / mean loss, first 100 updates : {np.mean(losses[:100]):.3f}")
print(f"마지막 100걸음 평균 손실 / mean loss, last 100 updates : {np.mean(losses[-100:]):.3f}")
print(f"걸린 시간 / time: {time.time() - t0:.0f} s")
# 기대값 / Expected   (같은 컴퓨터에서 다시 실행하면 같은 숫자. 다른 컴퓨터에서는 소수 셋째 자리가 다를 수 있습니다)
#                    (a re-run on the same computer gives the same numbers; another computer can differ in the third decimal)
#   처음 100걸음 / first 100  : 0.787
#   마지막 100걸음 / last 100  : 0.358
#   시간 / time: 이 노트북을 만든 2코어 CPU 에서 약 5~10초 / about 5 to 10 s on the 2-core CPU used to build this notebook
#   0 까지 내려가지 않습니다. 같은 x_t 를 만드는 (x0, ε) 짝이 여럿이라 ε 을 완전히 맞힐 수 없고, 가장 잘해도 그 짝들의 평균을 답하는 것이 최선입니다.
#   It does not reach 0: several pairs (x0, ε) make the same x_t, so ε cannot be recovered exactly; the best answer is the average over those pairs.

plt.figure(figsize=(5, 2.6))
plt.plot(losses, color=GRAY, lw=0.5, alpha=0.8)
k = 50; plt.plot(np.arange(k, len(losses) + 1), np.convolve(losses, np.ones(k) / k, mode="valid"), color=GREEN, lw=1.5)   # 50걸음 이동평균 / 50-update moving average
for s in (150, 600, 2000):
    plt.axvline(s, color=GOLD, lw=0.8, ls=":")
plt.xlabel("Adam updates"); plt.ylabel("noise MSE"); plt.ylim(0.25, 1.05); plt.title("Training loss (gold: the three checkpoints)", fontsize=10)
plt.tight_layout(); plt.show()

## §4 빈칸 2. 역방향 한 걸음, 그리고 표집 / Blank 2. One reverse step, then sampling
생성은 잡음 $x_T$ 에서 출발해 T 걸음을 거꾸로 걷습니다. 한 걸음은 「One DDPM Reverse Step」 의 두 줄을 하나로 쓴 것입니다. 예측한 잡음의 일부를 빼고, 이 걸음이 줄였던 신호를 되돌리고, 새 잡음을 더합니다.
*Generation starts from noise $x_T$ and walks the T steps backward. One step is the two lines of that slide written as one: remove part of the predicted noise, undo the shrink this step applied, add new noise.*

$$x_{t-1} = \underbrace{\frac{1}{\sqrt{\alpha_t}}\Big( x_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\, \hat\varepsilon \Big)}_{\text{평균 / the mean}} + \sigma_t z, \qquad \sigma_t = \sqrt{\beta_t}, \quad z \sim N(0, I)$$

**빈칸 하나.** 괄호 안과 되돌리기, 곧 평균 부분만 쓰는 함수입니다. 새 잡음 $\sigma_t z$ 는 아래 표집 함수가 더합니다. 확인 숫자는 「The Same Step, in Numbers」 의 것입니다(T = 1000, t = 500).
*The blank. A function for the mean part only: the bracket and the rescale. The new noise $\sigma_t z$ is added by the sampling function below. The check numbers are those of the slide "The Same Step, in Numbers" (T = 1000, t = 500).*

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   x_t      : 지금 걸음의 점들, 크기 (점 수, 2) / the points at the current step, shape (number of points, 2)
#   eps_hat  : 신경망이 예측한 잡음 ε̂, x_t 와 같은 크기 / the network's predicted noise ε̂, same shape as x_t
#   beta_t   : 이 걸음의 β_t (숫자 하나) / this step's β_t (one number)
#   alpha_t  : 이 걸음의 α_t = 1 − β_t / this step's α_t = 1 − β_t
#   abar_t   : 이 걸음까지의 누적 ᾱ_t / the running product ᾱ_t up to this step
#   mean     : 새 잡음을 더하기 전의 x_{t−1}   <- 여기가 빈칸
#              x_{t−1} before the new noise is added   <- this is the blank
#
# 참고 / Reference   torch.sqrt(a)   위 §2 와 같음 / as in §2 above
#   예 / e.g.   torch.sqrt(torch.tensor(0.25))  ->  tensor(0.5000)

def ddpm_mean(x_t, eps_hat, beta_t, alpha_t, abar_t):
    """「One DDPM Reverse Step」 의 평균 부분: 예측한 잡음의 일부를 빼고, 줄였던 신호를 되돌린다 / the mean part of that slide: remove part of the predicted noise, undo the shrink."""
    # TODO. 슬라이드의 괄호 안(x_t 에서 β_t/√(1 − ᾱ_t) · ε̂ 를 뺀 것)을 √α_t 로 나누세요. 한 줄입니다.
    #       나누는 것은 ᾱ_t 가 아니라 이 걸음의 α_t 입니다(함정 2).
    # TODO. Divide the slide's bracket (x_t minus β_t/√(1 − ᾱ_t) · ε̂) by √α_t. One line.
    #       The divisor is this step's α_t, not ᾱ_t (pitfall 2).
    mean = None

    if mean is None:
        raise NotImplementedError(BLANK_MSG)

    return mean

# 자가검증 / Check your work   「The Same Step, in Numbers」: T = 1000, t = 500, x_t = 1.0, ε̂ = 0.6
print(ddpm_mean(torch.tensor([[1.0]]), torch.tensor([[0.6]]), beta_t=torch.tensor(0.01004), alpha_t=torch.tensor(0.98996), abar_t=torch.tensor(0.07859)))
# 기대값 / Expected
#   tensor([[0.9988]])        슬라이드의 (1.0 − 0.01046 × 0.6) × 1.00505 = 0.9988 / the slide's 0.9988
#   tensor([[0.9937]]) 이면 √α_t 로 나누지 않았습니다(줄였던 신호를 되돌리는 부분이 빠짐).
#   tensor([[0.9937]]) means you did not divide by √α_t (the rescale that undoes the shrink is missing).
#   tensor([[3.5447]]) 이면 α_t 대신 ᾱ_t 로 나눴습니다. 한 걸음의 되돌리기에는 그 걸음의 α_t 가 들어갑니다.
#   tensor([[3.5447]]) means you divided by ᾱ_t instead of α_t. Undoing one step takes that step's α_t.
#   tensor([[0.9446]]) 이면 ε̂ 의 계수에 β_t/√(1 − ᾱ_t) 대신 √β_t 를 썼습니다. √β_t 는 새 잡음 σ_t 의 크기입니다.
#   tensor([[0.9446]]) means the coefficient on ε̂ was √β_t instead of β_t/√(1 − ᾱ_t). √β_t is the size σ_t of the new noise.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요. 이미 채웠다면 실행하지 마세요(내 함수를 덮어씁니다) / Only if stuck. If you already filled in the blank, skip it: it overwrites your function { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

# 다음 이름을 씁니다 / Names used below
#   x_t      : 지금 걸음의 점들, 크기 (점 수, 2) / the points at the current step, shape (number of points, 2)
#   eps_hat  : 신경망이 예측한 잡음 ε̂, x_t 와 같은 크기 / the network's predicted noise ε̂, same shape as x_t
#   beta_t   : 이 걸음의 β_t (숫자 하나) / this step's β_t (one number)
#   alpha_t  : 이 걸음의 α_t = 1 − β_t / this step's α_t = 1 − β_t
#   abar_t   : 이 걸음까지의 누적 ᾱ_t / the running product ᾱ_t up to this step
#   mean     : 새 잡음을 더하기 전의 x_{t−1}   <- 여기가 빈칸
#              x_{t−1} before the new noise is added   <- this is the blank
#
# 참고 / Reference   torch.sqrt(a)   위 §2 와 같음 / as in §2 above
#   예 / e.g.   torch.sqrt(torch.tensor(0.25))  ->  tensor(0.5000)

def ddpm_mean(x_t, eps_hat, beta_t, alpha_t, abar_t):
    """「One DDPM Reverse Step」 의 평균 부분: 예측한 잡음의 일부를 빼고, 줄였던 신호를 되돌린다 / the mean part of that slide: remove part of the predicted noise, undo the shrink."""
    mean = (x_t - beta_t / torch.sqrt(1 - abar_t) * eps_hat) / torch.sqrt(alpha_t)   # 빈칸이었던 줄. 예측한 잡음의 일부를 빼고(괄호), √α_t 로 되돌린다
                                                                                    # the blank: remove part of the predicted noise (bracket), then rescale by √α_t

    if mean is None:
        raise NotImplementedError(BLANK_MSG)

    return mean

# 자가검증 / Check your work   「The Same Step, in Numbers」: T = 1000, t = 500, x_t = 1.0, ε̂ = 0.6
print(ddpm_mean(torch.tensor([[1.0]]), torch.tensor([[0.6]]), beta_t=torch.tensor(0.01004), alpha_t=torch.tensor(0.98996), abar_t=torch.tensor(0.07859)))
# 기대값 / Expected
#   tensor([[0.9988]])        슬라이드의 (1.0 − 0.01046 × 0.6) × 1.00505 = 0.9988 / the slide's 0.9988
#   tensor([[0.9937]]) 이면 √α_t 로 나누지 않았습니다(줄였던 신호를 되돌리는 부분이 빠짐).
#   tensor([[0.9937]]) means you did not divide by √α_t (the rescale that undoes the shrink is missing).
#   tensor([[3.5447]]) 이면 α_t 대신 ᾱ_t 로 나눴습니다. 한 걸음의 되돌리기에는 그 걸음의 α_t 가 들어갑니다.
#   tensor([[3.5447]]) means you divided by ᾱ_t instead of α_t. Undoing one step takes that step's α_t.
#   tensor([[0.9446]]) 이면 ε̂ 의 계수에 β_t/√(1 − ᾱ_t) 대신 √β_t 를 썼습니다. √β_t 는 새 잡음 σ_t 의 크기입니다.
#   tensor([[0.9446]]) means the coefficient on ε̂ was √β_t instead of β_t/√(1 − ᾱ_t). √β_t is the size σ_t of the new noise.

In [ ]:
# 표집: 잡음에서 출발해 T 걸음을 거꾸로. 걸음마다 「예측 → 예측한 잡음의 일부 빼기(내 ddpm_mean) → 새 잡음 더하기」 / Sampling: start from noise and walk the T steps backward: predict, remove part of the predicted noise (your ddpm_mean), add new noise
@torch.no_grad()
def sample_ddpm(model, n, gen):
    x = torch.randn(n, 2, generator=gen)                                 # x_T: 순수 잡음 / pure noise
    for t in range(T, 0, -1):                                            # t = T, T−1, …, 1
        eps_hat = model(x, torch.full((n,), t))                          # 이 걸음의 잡음 예측 / predict this step's noise
        x = ddpm_mean(x, eps_hat, betas[t], alphas[t], abar[t])          # 평균 부분 / the mean part
        if t > 1:
            x = x + torch.sqrt(betas[t]) * torch.randn(n, 2, generator=gen)   # σ_t z. 마지막 걸음에는 더하지 않는다 / σ_t z; none on the last step
    return x

def share_near_center(pts, radius=1.0):
    """어느 중심에서든 radius 안에 든 점의 비율 / the share of points within radius of some center."""
    d = torch.cdist(pts, CENTERS).min(1).values                          # 점마다 가장 가까운 중심까지의 거리 / distance to the nearest center per point
    return (d < radius).float().mean().item()

fig, axs = plt.subplots(1, 3, figsize=(8, 2.7))
print("학습 걸음 / updates   중심 1.0 안의 표본 비율 / share of samples within 1.0 of a center")
for ax, step in zip(axs, (150, 600, 2000)):
    net = Net(); net.load_state_dict(ckpt[step])                         # 그 시점의 가중치 / the weights at that point of training
    pts = sample_ddpm(net, 600, torch.Generator().manual_seed(1000 + step))
    print(f"{step:6d}                {100 * share_near_center(pts):.0f} %")
    ax.scatter(pts[:, 0], pts[:, 1], s=3, color=GREEN, alpha=0.55)
    ax.scatter(CENTERS[:, 0], CENTERS[:, 1], marker="+", s=70, color=GOLD, linewidths=1.6)
    ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], title=f"{step} updates"); ax.set_aspect("equal")
plt.suptitle("600 DDPM samples from the three checkpoints (gold + = true centers)", fontsize=10); plt.tight_layout(); plt.show()
# 기대값 / Expected   (슬라이드 「As the Loss Falls, Do the Samples Match the Data?」 의 표본 세 칸이 이 셀의 출력이고, 슬라이드는 진짜 자료 칸과 퍼짐을 더합니다 / the three sample panels on that slide are this cell's output; the slide adds the real data and the spread)
#    150 updates : 21 %   아직 퍼진 구름 / still a spread-out cloud
#    600 updates : 81 %   네 군집, 사이에 흘린 점 / four clusters, with points spilled between them
#   2000 updates : 98 %   더 조밀 / tighter

**무엇을 보았나.** 손실은 0.79 에서 0.36 으로 내려가고 멈춥니다. 같은 $x_t$ 를 만들 수 있는 $(x_0, \varepsilon)$ 짝이 여럿이라 잡음을 완전히 맞힐 수는 없고, 남은 0.36 은 그 「맞힐 수 없는 몫」입니다. 그래도 표본은 150 걸음의 퍼진 구름에서 600 걸음의 네 군집으로, 2,000 걸음의 더 조밀한 군집으로 좋아집니다. 「As the Loss Falls, Do the Samples Match the Data?」 의 답은 「대체로 그렇다, 다만 손실의 바닥은 0 이 아니다」 입니다.
*What did we see. The loss falls from 0.79 to 0.36 and stops. Several pairs $(x_0, \varepsilon)$ can produce the same $x_t$, so the noise cannot be guessed exactly; the remaining 0.36 is that unguessable share. The samples still improve: a spread-out cloud at 150 updates, four clusters at 600, tighter clusters at 2,000. The answer to that slide's question is "mostly yes, but the floor of the loss is not 0".*

표집은 100 번의 신경망 계산입니다(「The Problem: Sampling Takes Too Long」). 점 600개를 뽑는 데도 몇 초가 걸렸습니다. 다음 절이 그 걸음을 줄입니다.
*Sampling took 100 network passes (that slide). Even 600 points took a few seconds. The next section cuts the steps.*

## §5 DDIM 으로 걸음 줄이기 / DDIM with fewer steps
「DDIM: Can We Generate With Fewer Steps?」: 같은 신경망, 같은 학습, 걷는 길만 다릅니다. 한 걸음은 두 줄입니다. 먼저 「The Network Learns to Predict the Noise」 의 식으로 $\hat x_0$ 을 짐작하고, 그 $\hat x_0$ 과 같은 $\hat\varepsilon$ 으로 다음 걸음 s 의 $x_s$ 를 §2 의 한 줄 공식으로 다시 만듭니다. 새 잡음은 더하지 않습니다.
*That slide: the same network, the same training, only a different path. One step is two lines. First guess $\hat x_0$ with the formula from "The Network Learns to Predict the Noise", then rebuild the next step's $x_s$ from that $\hat x_0$ and the same $\hat\varepsilon$ with the one-line formula of §2. No new noise.*

$$\hat x_0 = \frac{x_t - \sqrt{1 - \bar\alpha_t}\,\hat\varepsilon}{\sqrt{\bar\alpha_t}}, \qquad x_s = \sqrt{\bar\alpha_s}\, \hat x_0 + \sqrt{1 - \bar\alpha_s}\, \hat\varepsilon \quad (s < t)$$

s 를 t − 1 로 두면 걸음이 100 개이고, 걸음을 건너뛰면 그만큼 줄어듭니다. 아래 셀은 같은 시작 잡음 600개를 5 · 10 · 25 · 100 걸음으로 걷습니다.
*With s = t − 1 there are 100 steps; skipping steps cuts that number. The cell below walks the same 600 starting noises in 5, 10, 25 and 100 steps.*

**코드를 돌리기 전에** 예측을 적으세요. 몇 걸음이면 100 걸음과 거의 같아질까요? 걸음이 너무 적을 때 먼저 잃는 것은 네 군집이라는 큰 구조일까요, 군집 안의 모양이라는 세부일까요?
*Before running, write a prediction: at how many steps will the result be nearly the same as 100 steps? With too few steps, what is lost first, the big structure (four clusters) or the detail (the shape inside a cluster)?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_3 = ""

In [ ]:
# DDIM 표집 / DDIM sampling
@torch.no_grad()
def sample_ddim(model, x_T, steps):
    """새 잡음 없이, T 에서 0 까지 `steps` 개의 걸음만 밟는다 / No new noise; visit only `steps` of the steps from T down to 0."""
    x = x_T.clone()
    ts = np.unique(np.round(np.linspace(T - 1, 0, steps)).astype(int))[::-1] + 1   # 0 … T−1 에서 고르게 고른 뒤 1 을 더해 1 … T 로. 큰 t 부터 / pick evenly in 0 … T−1, add 1 to get 1 … T; largest t first
    for j, t in enumerate(ts):
        s = int(ts[j + 1]) if j + 1 < len(ts) else 0                           # 다음 걸음. 마지막은 0 (데이터) / the next step; the last one is 0 (data)
        eps_hat = model(x, torch.full((len(x),), int(t)))
        x0_hat = (x - torch.sqrt(1 - abar[t]) * eps_hat) / torch.sqrt(abar[t])   # 첫 줄: x̂0 짐작 / line 1: guess x̂0
        x = torch.sqrt(abar[s]) * x0_hat + torch.sqrt(1 - abar[s]) * eps_hat     # 둘째 줄: §2 의 한 줄로 x_s 를 다시 만든다 (ᾱ_0 = 1) / line 2: rebuild x_s with the §2 line (ᾱ_0 = 1)
    return x

x_T = torch.randn(600, 2, generator=torch.Generator().manual_seed(7))     # 네 번 모두 같은 시작 잡음 / the same starting noise for all four runs
fig, axs = plt.subplots(1, 4, figsize=(10, 2.7))
print("걸음(= 신경망 계산 횟수) / steps (= network passes)   중심 1.0 안의 비율 / share within 1.0 of a center")
for ax, k in zip(axs, (5, 10, 25, 100)):
    pts = sample_ddim(model, x_T, k)
    print(f"{k:5d}                                           {100 * share_near_center(pts):.0f} %")
    ax.scatter(pts[:, 0], pts[:, 1], s=3, color=GREEN, alpha=0.55)
    ax.scatter(CENTERS[:, 0], CENTERS[:, 1], marker="+", s=70, color=GOLD, linewidths=1.6)
    ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], title=f"{k} steps"); ax.set_aspect("equal")
plt.suptitle("Same network, same 600 starting noises, DDIM with fewer steps", fontsize=10); plt.tight_layout(); plt.show()

again = sample_ddim(model, x_T, 25)                                          # 같은 x_T 로 한 번 더 / once more from the same x_T
print(f"같은 x_T 로 다시 뽑은 25걸음 표본과의 최대 차이 / largest difference from a second 25-step run with the same x_T: {(again - sample_ddim(model, x_T, 25)).abs().max():.1e}")
# 기대값 / Expected   (슬라이드 「What Happens When We Skip Steps?」 의 네 칸이 이 셀의 출력이고, 슬라이드는 진짜 자료 칸과 퍼짐을 더합니다 / the four panels on that slide are this cell's output; the slide adds the real data and the spread)
#     5 steps : 79 %     군집 사이에 흘린 점이 많음 / many points spilled between the clusters
#    10 steps : 90 %
#    25 steps : 94 %     100 걸음과 거의 같음 / nearly the same as 100 steps
#   100 steps : 96 %
#   최대 차이 0.0e+00: 새 잡음이 없으므로 같은 x_T 는 같은 점을 줍니다(「DDIM: Can We Generate With Fewer Steps?」 의 표).
#   largest difference 0.0e+00: with no new noise, the same x_T gives the same point (the table on that slide).

**무엇을 보았나.** 5 걸음은 큰 구조(네 군집)는 있지만 군집 사이에 점을 흘리고, 10 걸음에서 25 걸음 사이에서 꺾여 25 걸음이면 100 걸음과 거의 같습니다. 「What Happens When We Skip Steps?」 의 두 질문에 이 자료가 주는 답: 품질은 10 ~ 25 걸음 사이에서 꺾이고, 먼저 잃는 것은 세부(군집 안의 모양)입니다. 걸음은 곧 신경망 계산 횟수이니 초가 아니라 횟수를 세세요. 그리고 §3 을 다시 돌리지 않았습니다. 함정 3: DDIM 을 위해 다시 학습하지 않습니다.
*What did we see. 5 steps keep the big structure (four clusters) but spill points between them; quality bends between 10 and 25 steps, and 25 is nearly the same as 100. This data's answers to the two questions on that slide: quality bends between 10 and 25 steps, and detail (the shape inside a cluster) is lost first. Steps are network passes, so count passes, not seconds. And §3 was not re-run. Pitfall 3: no retraining for DDIM.*

## (선택) 흐름 매칭 열 줄 / (Optional) The flow-matching recipe in ten lines
「Optional D · Flow Matching」: DDIM 이 새 잡음 없이 한 길을 걷는다면, 그 길의 속도를 바로 배우면 어떨까요. 잡음 z 와 데이터 x 를 직선으로 잇고($x_t = (1-t)\,z + t\,x$, t 는 0 에서 1), 그 직선의 속도 $x - z$ 를 맞히는 회귀입니다. 손실의 모양은 §3 과 같습니다. 닫힌 식으로 목표를 만들고, 제곱 오차를 줄입니다. 표본은 z 에서 출발해 배운 속도로 몇 걸음 걸으면 됩니다.
*That slide: if DDIM walks one path with no new noise, why not learn the path's velocity directly? Join noise z and data x by a straight line ($x_t = (1-t)\,z + t\,x$, t from 0 to 1) and regress on that line's velocity $x - z$. The loss has the same shape as §3: build a target in closed form, minimize a squared error. To sample, start at z and walk a few steps with the learned velocity.*

같은 Net 을 쓰되 t 를 0 ~ 1 사이의 수로 넣습니다. 빈칸은 없습니다.
*The same Net, with t fed as a number between 0 and 1. No blank.*

In [ ]:
# 흐름 매칭 열 줄 / Flow matching in ten lines
class VelocityNet(Net):
    def forward(self, x, t01):                                            # t01: 0 ~ 1 사이의 수 / a number between 0 and 1
        return self.net(torch.cat([x, t_features(t01 * T + 1)], 1))      # §3 의 t 펴기를 그대로 빌려 쓴다 / reuse the t-spreading of §3

gen_fm = torch.Generator().manual_seed(29); torch.manual_seed(29)
vnet = VelocityNet(); opt_fm = torch.optim.Adam(vnet.parameters(), lr=1e-3)
for step in range(2000):
    x1 = data[torch.randint(0, len(data), (512,), generator=gen_fm)]      # 데이터 x / data x
    z = torch.randn(x1.shape, generator=gen_fm)                           # 잡음 z / noise z
    t01 = torch.rand(len(x1), generator=gen_fm)                           # t ~ U[0, 1]
    x_t = (1 - t01[:, None]) * z + t01[:, None] * x1                      # 직선 위의 점 / a point on the straight line
    loss_fm = (vnet(x_t, t01) - (x1 - z)).square().mean()                 # 목표 속도 x − z 에 대한 제곱 오차 / squared error on the target velocity x − z
    opt_fm.zero_grad(); loss_fm.backward(); opt_fm.step()

with torch.no_grad():                                                     # 표집: z 에서 출발해 10 걸음 / sampling: from z in 10 steps
    x = torch.randn(600, 2, generator=torch.Generator().manual_seed(7))
    for i in range(10):
        x = x + 0.1 * vnet(x, torch.full((600,), i / 10))                 # dx/dt = v 를 폭 0.1 로 / dx/dt = v with step 0.1
print(f"마지막 손실 / final loss: {loss_fm.item():.3f}   10 걸음 표본 중 중심 1.0 안 / 10-step samples within 1.0 of a center: {100 * share_near_center(x):.0f} %")
# 기대값 / Expected   마지막 손실 약 2.3, 중심 1.0 안 약 90 % (10 걸음) / final loss about 2.3, about 90 % within 1.0 (10 steps)
#   손실의 크기는 §3 과 비교하지 마세요. 목표가 다릅니다(잡음 ε 이 아니라 속도 x − z). 같은 종류의 회귀라는 점만 보면 됩니다.
#   Do not compare the loss size with §3: the target differs (the velocity x − z, not the noise ε). The point is that it is the same kind of regression.

## 정리. 예측 다시 읽기 / Wrap-up. Re-read your predictions
1. **§1.** 50 걸음쯤이면 군집이 흐려지고 100 걸음이면 N(0, I) 에 가깝습니다. 점들의 퍼짐은 식 $\sqrt{\bar\alpha_t \cdot \text{데이터 분산} + (1 - \bar\alpha_t)}$ 가 말하는 값과 같았습니다. 배운 것은 없습니다.
   *§1. The clusters blur by about step 50 and are near N(0, I) at step 100. The spread matched the formula. Nothing was learned.*
2. **§2.** 한 줄 공식의 두 계수는 $\sqrt{\bar\alpha_t}$ 와 $\sqrt{1 - \bar\alpha_t}$ 입니다. 몸풀기의 픽셀(0.0)과 한 걸음씩 걸은 §1 의 퍼짐이 같은 식에서 나왔습니다.
   *§2. The two coefficients are $\sqrt{\bar\alpha_t}$ and $\sqrt{1 - \bar\alpha_t}$. The warm-up pixel (0.0) and the spread of the step-by-step walk in §1 came from the same line.*
3. **§3 예측 2.** 손실은 0.79 에서 0.36 으로 내려가고 0 에는 가지 않습니다. 같은 $x_t$ 를 만드는 $(x_0, \varepsilon)$ 짝이 여럿이라 잡음은 평균으로만 맞힐 수 있습니다.
   *§3 prediction 2. The loss falls from 0.79 to 0.36 and never reaches 0, because several pairs $(x_0, \varepsilon)$ make the same $x_t$ and the noise can only be guessed on average.*
4. **§4.** 역방향 한 걸음은 예측한 잡음의 일부 빼기, 되돌리기, 새 잡음 더하기입니다. 슬라이드의 숫자(0.9988)가 그대로 나왔고, 표본은 150 → 600 → 2,000 걸음에서 21 → 81 → 98 % 가 중심 1.0 안에 들었습니다.
   *§4. One reverse step is remove, rescale, re-randomize. The slide's number (0.9988) came out as it is, and 21, 81 and 98 % of the samples landed within 1.0 of a center at 150, 600 and 2,000 updates.*
5. **§5 예측 3.** 같은 신경망으로 5 · 10 · 25 · 100 걸음: 79 · 90 · 94 · 96 %. 품질은 10 ~ 25 걸음 사이에서 꺾이고, 먼저 잃는 것은 세부입니다. 새 잡음이 없어 같은 $x_T$ 는 같은 점을 줍니다.
   *§5 prediction 3. 79, 90, 94 and 96 % at 5, 10, 25 and 100 steps with the same network. Quality bends between 10 and 25 steps, and detail goes first. With no new noise, the same $x_T$ gives the same point.*

함정과 짝(「Four Ways to Get This Wrong」): 1번 「모형이 잡음 더하기를 배운다」는 §1(요리법은 우리가 정함), 2번 「β_t 와 ᾱ_t 를 섞음」은 §2 · §4 의 빈칸(한 줄 공식은 ᾱ_t, 한 걸음 되돌리기는 α_t), 3번 「DDIM 을 위해 다시 학습」은 §5(같은 model), 4번 「베끼기 점검 없이 좋은 표본을 믿음」은 본 덱 「Another Problem: Can It Copy Training Data?」 의 그림이 이 노트북의 설정에서 학습 점 8개로 한 실험입니다. 과제 3 에서 그 점검을 직접 합니다.
*Pitfalls on the slide: 1 (the model learns to add noise) is §1, where we chose the recipe; 2 (mixing up β_t and ᾱ_t) is the two blanks of §2 and §4 (the one-line formula takes ᾱ_t, undoing one step takes α_t); 3 (retraining for DDIM) is §5, the same model; 4 (trusting good samples without a copy check) is the figure on the slide "New Problem ②  With Little Data, It Can Copy", an experiment in this notebook's setup with 8 training points. You run that check yourself in Assignment 3.*

In [ ]:
# 실행 환경 기록 / Record the runtime
import sys, platform, datetime
print("Runtime :", platform.platform(), "| Python", sys.version.split()[0])
print("torch   :", torch.__version__, "| threads", torch.get_num_threads(), "|", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Run at  :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))